Questions:1

1. 𝐅𝐢𝐧𝐝 𝐭𝐡𝐞 𝐞𝐚𝐫𝐥𝐢𝐞𝐬𝐭 𝐚𝐧𝐝 𝐥𝐚𝐭𝐞𝐬𝐭 𝐭𝐢𝐦𝐞𝐬𝐭𝐚𝐦𝐩𝐬 𝐢𝐧 𝐭𝐡𝐞 𝐝𝐚𝐭𝐚𝐬𝐞𝐭.

2. 𝐂𝐨𝐮𝐧𝐭 𝐭𝐡𝐞 𝐧𝐮𝐦𝐛𝐞𝐫 𝐨𝐟 𝐚𝐜𝐭𝐢𝐯𝐢𝐭𝐢𝐞𝐬 𝐩𝐞𝐫 𝐮𝐬𝐞𝐫.

3.𝐂𝐚𝐥𝐜𝐮𝐥𝐚𝐭𝐞 𝐭𝐡𝐞 𝐭𝐢𝐦𝐞 𝐝𝐮𝐫𝐚𝐭𝐢𝐨𝐧 𝐛𝐞𝐭𝐰𝐞𝐞𝐧 𝐜𝐨𝐧𝐬𝐞𝐜𝐮𝐭𝐢𝐯𝐞 𝐚𝐜𝐭𝐢𝐯𝐢𝐭𝐢𝐞𝐬 𝐟𝐨𝐫 𝐞𝐚𝐜𝐡 𝐮𝐬𝐞𝐫.

Sample Dataset :

columns = ["user_id", "timestamp"]

data= [("user1", "2023-08-21 10:00:00"),
       
("user2", "2023-08-21 11:30:00"),

("user1", "2023-08-21 12:15:00"),

("user3", "2023-08-21 13:45:00"),

("user2", "2023-08-21 14:30:00"),

("user1", "2023-08-21 15:00:00")]




In [0]:
#QUESTIONS 1

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, min, max, count, unix_timestamp, lag
from pyspark.sql.window import Window

# Initialize SparkSession
spark = SparkSession.builder.appName("ActivityAnalysis").getOrCreate()

# Sample dataset for Question 1
columns = ["user_id", "timestamp"]
data = [
    ("user1", "2023-08-21 10:00:00"),
    ("user2", "2023-08-21 11:30:00"),
    ("user1", "2023-08-21 12:15:00"),
    ("user3", "2023-08-21 13:45:00"),
    ("user2", "2023-08-21 14:30:00"),
    ("user1", "2023-08-21 15:00:00")
]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert timestamp to timestamp type
df = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

# 1. Find earliest and latest timestamps
earliest_time = df.agg(min("timestamp")).collect()[0][0]
latest_time = df.agg(max("timestamp")).collect()[0][0]
print("Earliest Timestamp:", earliest_time)
print("Latest Timestamp:", latest_time)

# 2. Count the number of activities per user
activity_count = df.groupBy("user_id").agg(count("timestamp").alias("activity_count"))
activity_count.show()

# 3. Calculate time duration between consecutive activities for each user
window_spec = Window.partitionBy("user_id").orderBy("timestamp")
df = df.withColumn("prev_timestamp", lag("timestamp").over(window_spec))
df = df.withColumn("time_diff", (unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")).cast("int"))
df.show()

Earliest Timestamp: 2023-08-21 10:00:00
Latest Timestamp: 2023-08-21 15:00:00
+-------+--------------+
|user_id|activity_count|
+-------+--------------+
|  user1|             3|
|  user2|             2|
|  user3|             1|
+-------+--------------+

+-------+-------------------+-------------------+---------+
|user_id|          timestamp|     prev_timestamp|time_diff|
+-------+-------------------+-------------------+---------+
|  user1|2023-08-21 10:00:00|               null|     null|
|  user1|2023-08-21 12:15:00|2023-08-21 10:00:00|     8100|
|  user1|2023-08-21 15:00:00|2023-08-21 12:15:00|     9900|
|  user2|2023-08-21 11:30:00|               null|     null|
|  user2|2023-08-21 14:30:00|2023-08-21 11:30:00|    10800|
|  user3|2023-08-21 13:45:00|               null|     null|
+-------+-------------------+-------------------+---------+



Questions:2

1. Count the number of actions

2: What are the unique actions recorded in the dataset?

3: Calculate the time duration between consecutive activities for each user


data = [

    (1, "login", "2023-08-20 10:23:45"),

    (2, "view", "2023-08-20 11:15:30"),

    (1, "purchase", "2023-08-20 12:45:18"),

    (3, "view", "2023-08-20 13:30:22")

]

columns = ["user_id", "action", "timestamp"]

In [0]:
from pyspark.sql.functions import countDistinct

# Sample dataset for Question 2
data = [
    (1, "login", "2023-08-20 10:23:45"),
    (2, "view", "2023-08-20 11:15:30"),
    (1, "purchase", "2023-08-20 12:45:18"),
    (3, "view", "2023-08-20 13:30:22")
]
columns = ["user_id", "action", "timestamp"]

# Create DataFrame
df2 = spark.createDataFrame(data, columns)

# Convert timestamp to timestamp type
df2 = df2.withColumn("timestamp", col("timestamp").cast("timestamp"))

# 1. Count the number of actions
total_actions = df2.count()
print("Total Actions:", total_actions)

# 2. Find unique actions
unique_actions = df2.select("action").distinct()
print("Unique Actions:")
unique_actions.show()

# 3. Calculate time duration between consecutive activities for each user
window_spec = Window.partitionBy("user_id").orderBy("timestamp")
df2 = df2.withColumn("prev_timestamp", lag("timestamp").over(window_spec))
df2 = df2.withColumn("time_diff", (unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")).cast("int"))
df2.show()


Total Actions: 4
Unique Actions:
+--------+
|  action|
+--------+
|   login|
|    view|
|purchase|
+--------+

+-------+--------+-------------------+-------------------+---------+
|user_id|  action|          timestamp|     prev_timestamp|time_diff|
+-------+--------+-------------------+-------------------+---------+
|      1|   login|2023-08-20 10:23:45|               null|     null|
|      1|purchase|2023-08-20 12:45:18|2023-08-20 10:23:45|     8493|
|      2|    view|2023-08-20 11:15:30|               null|     null|
|      3|    view|2023-08-20 13:30:22|               null|     null|
+-------+--------+-------------------+-------------------+---------+

